# Regressionsbäume und Ensembles — Konzepte

Lies die Idee, die Formel und das kleine Beispiel in dieser Reihenfolge. Der Demonstrationscode wird bereitgestellt.

## Libraries and settings

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score
from sklearn.inspection import permutation_importance

## 1. Ein Regressionsbaum sagt einen Mittelwert voraus

Ein Baum stellt Ja-/Nein-Fragen wie „Motorgröße ≤ 2 Liter?“. Jede Antwort führt uns zu einem weiteren Knoten. Der letzte Knoten ist ein Blatt. Seine Vorhersage ist der Mittelwert der Zielvariable der Trainingszeilen in diesem Blatt. Ein Baum teilt den Eingaberaum in Regionen auf, mit einer konstanten Vorhersage pro Region.

Um einen Split auszuwählen, berechne die quadrierten Fehler um den jeweiligen Mittelwert der beiden Kindknoten:
$$\mathrm{SSE}{\mathrm{split}}=\sum{i\in\mathrm{links}}(y_i-\bar y_{\mathrm{links}})^2+\sum_{i\in\mathrm{rechts}}(y_i-\bar y_{\mathrm{rechts}})^2.$$
Probiere mögliche Merkmale und Schwellenwerte aus; wähle die kleinste Gesamt-SSE. Wiederhole dies in jedem Kindknoten (rekursives, gieriges Aufteilen). Für die Zielwerte 10, 12, 14, 30, 34 ergibt ein Split nach 14 die Vorhersagen 12 und 32 und eine Gesamt-SSE von 16, verglichen mit 496 vor dem Split.

In [2]:
y = np.array([10, 12, 14, 30, 34])
left = y[:3]
right = y[3:]
print('Parent SSE:', np.sum((y - y.mean()) ** 2))
print('Leaf means:', left.mean(), right.mean())
print('Split SSE:', np.sum((left - left.mean()) ** 2) + np.sum((right - right.mean()) ** 2))

Parent SSE: 496.0
Leaf means: 12.0 32.0
Split SSE: 16.0


## 2. Grösse, Hyperparameter und Pruning

Tiefe Bäume können Rauschen lernen. Ein **Hyperparameter** ist eine Einstellung, die wir vor dem Anpassen auswählen.

| Parameter | Bedeutung | Auswirkung einer Ehöhung |
|---|---|---|
| `max_depth` | maximale Tiefe| ermöglicht einen flexibleren Baum |
| `min_samples_split` | benötigte Zeilen, um einen Knoten zu teilen | verhindert kleine Splits |
| `min_samples_leaf` | minimale Anzahl an Zeilen pro Blatt | glättet die Vorhersagen |
| `max_leaf_nodes` | maximale Anzahl an Blättern | ermöglicht mehr Regionen |
| `max_features` | bei einem Split berücksichtigte Eingaben | weniger Zufälligkeit, wenn mehr berücksichtigt werden|
| `ccp_alpha` | Pruning-Strafterm | kleinerer Baum|
| `criterion` | Split-Maß | Regression: quadratischer Fehler; Klassifikation: Gini/Entropie |

**Pruning** entfernt Äste, deren Nutzen im Vergleich zu einer Gebühr pro Blatt zu gering ist:
$$\mathrm{Pruning\text{-}Zielfunktion}=\mathrm{Baumfehler}+\alpha\times\mathrm{Anzahl\ der\ Blätter}.$$
Hier ist $\alpha$ der Strafterm. ccp_alpha=0 fügt keine Gebühr hinzu; größere Werte können Äste entfernen. Scikit-learn drückt den Regressionsfehler auf einer Pro-Beobachtung-Skala aus, daher hängen die Alpha-Werte von den Einheiten der Zielvariable ab. Wähle die Einstellungen mit CV auf den Trainingsdaten.

## 3. Bagging und Out-of-Bag-Zeilen

**Bagging** passt Bäume auf Bootstrap-Stichproben an: Trainingszeilen werden mit Zurücklegen gezogen, sodass einige wiederholt vorkommen und andere ausgelassen werden. Die Vorhersagen werden gemittelt:
$$\hat y_{\mathrm{bag}}=\frac1B\sum_{b=1}^{B}\hat y_b.$$
$B$ ist die Anzahl der Bäume. Die Vorhersagen 8, 10, 9 ergeben einen Mittelwert von 9.

Bei einer großen Stichprobe werden ungefähr 37 % der Zeilen aus der Bootstrap-Stichprobe eines bestimmten Baums ausgelassen. Diese **Out-of-Bag-(OOB-)**Zeilen ermöglichen eine interne Evaluation mithilfe von Bäumen, die die jeweilige Zeile nicht gesehen haben. Die OOB-Evaluation ersetzt nicht den abschließenden Test.

Mittelwertbildung reduziert Instabilität. Bei gleicher Baumvarianz $\sigma^2$ und gemeinsamer Korrelation $\rho$:
$$\mathrm{Var}(\mathrm{Mittelwert})=\rho\sigma^2+\frac{(1-\rho)\sigma^2}{B}.$$
Nur der unabhängige Teil wird kleiner, wenn wir Bäume hinzufügen. Für $\sigma^2=1$, $\rho=0{,}2$ und $B=100$ beträgt die Varianz 0,208. Keine Herleitung ist erforderlich.

## 4. Random Forest

Ein Random Forest ergänzt Bagging um **zufällige Teilmengen von Merkmalen bei jedem Split**. Dadurch werden die Bäume weniger ähnlich, sodass die Mittelwertbildung stärker helfen kann. Verwende n_estimators für die Anzahl der Bäume und max_features für die Eingaben, die bei jedem Split berücksichtigt werden.

In scikit-learn verwendet ein Regressions-Forest standardmäßig alle Eingaben (max_features=1.0); ein Klassifikations-Forest verwendet standardmäßig "sqrt". Mehr Bäume stabilisieren die Ergebnisse, bei zunehmendem Rechenaufwand und abnehmendem zusätzlichem Nutzen.

Die Feature Importance fragt, welche Eingaben bei der Vorhersage helfen. Die Impurity Importance kann bestimmte Eingaben unfair bevorzugen. Die **Permutation Importance** mischt ein Merkmal zufällig und misst die Abnahme der Vorhersageleistung. Importance ist kein kausaler Effekt.

## 5. Gradient Boosting und Vergleich

Boosting baut kleine Bäume nacheinander auf. Beim quadratischen Verlust lernt jeder neue Baum die aktuellen Residuen:
$$r_i=y_i-F_{m-1}(x_i),\qquad F_m(x)=F_{m-1}(x)+\nu h_m(x).$$
$F_{m-1}$ ist das aktuelle Modell, $h_m$ der nächste Baum und $\nu$ die Lernrate. Wenn die aktuelle Vorhersage 7 beträgt, der nächste Baum ein Residuum von 3 vorhersagt und $\nu=0{,}1$ ist, beträgt die aktualisierte Vorhersage $7+0{,}1\times3=7{,}3$. Andere differenzierbare Verlustfunktionen verwenden negative Gradienten (Pseudo-Residuen).

| Modell | Hauptidee | Haupteinstellungen|
|---|---|---|
| Lineare Regression| gewichtete Summe der Eingaben | ausgewählte Eingaben |
| Einzelner Baum | verständliche Regeln und konstante Blattvorhersagen | Tiefe, Pruning |
| Random Forest | Mittelwert vieler unterschiedlicher Bäume | Bäume, zufällige Merkmale |
| Boosting | Fehler Schritt für Schritt korrigieren | Lernrate, Bäume, Tiefe |

Bagging reduziert hauptsächlich die Varianz; Boosting reduziert systematische Fehler Schritt für Schritt und kann sowohl Bias als auch Varianz beeinflussen. Vergleiche die Validierungsergebnisse: Keine Methode gewinnt immer.

In [3]:
old_prediction = 7
residual_prediction = 3
learning_rate = 0.1
print('Updated prediction:', old_prediction + learning_rate * residual_prediction)

Updated prediction: 7.3


### Jupyter notebook — footer info

Include this information when submitting your notebook.

In [4]:
import platform
from datetime import datetime
print(platform.platform())
print("Python:", platform.python_version())
print("Date:", datetime.now().isoformat(timespec="seconds"))

Linux-6.8.0-1064-azure-x86_64-with-glibc2.41
Python: 3.11.17
Date: 2026-10-20T05:37:19


| Methode | Was passiert? | Wie arbeiten die Bäume? | Hauptziel | Merksatz |
|---|---|---|---|---|
| **Single Regression Tree** | Ein einzelner Entscheidungsbaum teilt die Daten schrittweise auf | Nur **1 Baum** | Einfache und verständliche Regeln finden | **Ein Baum** |
| **Bagging** | Viele Bäume werden auf unterschiedlichen Bootstrap-Stichproben trainiert | Bäume werden **unabhängig/parallel** trainiert, danach werden die Vorhersagen gemittelt | Varianz reduzieren und Vorhersagen stabiler machen | **Viele Bäume + Mittelwert** |
| **Random Forest** | Wie Bagging, aber bei jedem Split wird nur eine **zufällige Auswahl von Features** betrachtet | Viele unterschiedliche Bäume werden trainiert, danach werden die Vorhersagen gemittelt | Bäume weniger ähnlich machen und dadurch stabilere Vorhersagen erhalten | **Bagging + zufällige Features** |
| **Boosting** | Viele kleine Bäume werden **nacheinander** trainiert | Jeder neue Baum versucht, die Fehler der bisherigen Bäume zu korrigieren | Fehler Schritt für Schritt reduzieren | **Jeder neue Baum korrigiert Fehler** |